In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas_datareader import data as pdr
from datetime import datetime, timedelta
import statsmodels.api as sm
import os
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [ ]:
class Utils:
  @staticmethod
  def read_data(ticker, start, end, constrains):
    try:
      t = yf.Ticker(ticker)
      history = t.history(start=start, end=end, interval="1mo")[['Close', 'Volume']]
      cons_check = Utils.check_constrains(history["Close"].mean(), history["Volume"].mean(), constrains)
      close = history["Close"]


      if not cons_check:
        return False

      shares = t.info.get("sharesOutstanding")

      market_cap = close.values[-1] * shares

      annual_bs = t.balance_sheet
      annual_bv = annual_bs.loc['Stockholders Equity'].iloc[0]

      bm = annual_bv / market_cap


    except Exception as e:
      raise Exception(f"Error retrieving data for {ticker}: {e}")

    else:
      returns = close.pct_change().values
      momentum = 1
      for i in range(1, 12):
        momentum *= (1 + returns[-i])

      momentum -= 1
      return returns, market_cap, bm, momentum

  def check_constrains(close_mean, volume_mean, constrains):
    volume_mean = volume_mean / 1000000

    if constrains is None:
      return True

    if close_mean < constrains["min_price"] or volume_mean < constrains["min_volume"]:
      return False

    return True

  @staticmethod
  def compute_z_scores(x):
    mu = np.mean(x)
    sigma = np.std(x)

    z = (x - mu) / sigma

    return z, sigma

  @staticmethod
  def dot_product(A, B):
    return np.sum(A * B, axis=1).reshape(-1, 1)



In [ ]:
class DataReader:
  def __init__(self, tickers, start, end, constrains, reader_params):
    self.start = start
    self.end = end
    self.tickers = tickers
    self.reader_params = reader_params
    self.factors = None
    self.constrains = constrains
    self.market_data = None


  def generate_data(self):
    self.factors = self.read_factors(reader_params)
    data = self.read_market_data(
        self.tickers, self.start,
        self.end, self.constrains
    )
    self.market_data, self.BM, self.m_caps, self.momentum = data

  def read_factors(self, reader_params):
    factors = pdr.DataReader(*self.reader_params)
    factor_data = factors[0]
    factor_data = factor_data[self.start:self.end]
    return factor_data


  def read_market_data(self, tickers, start, end, constrains):
    price_data = {}
    market_caps = {}
    momentum = {}
    BMs = {}

    for ticker in tickers:
      res = Utils.read_data(ticker, start, end, constrains)

      if not res:
        continue

      returns, market_cap, bm, mm = res
      price_data[ticker] = returns
      market_caps[ticker] = market_cap
      BMs[ticker] = bm
      momentum[ticker] = mm

    m_caps = pd.Series(market_caps)
    BM = pd.Series(BMs)
    mm = pd.Series(momentum)

    for k in price_data.keys():
      print(f"{k}: {len(price_data[k])}")

    m_data = pd.DataFrame(price_data, index=self.factors.index).dropna()


    return m_data, BM, m_caps, mm

  @property
  def datastore(self):
    return {
      "BM": self.BM,
      "factors": self.factors,
      "market_data": self.market_data,
      "size": np.log(self.m_caps),
      "momentum": self.momentum
    } if self.market_data is not None else None



In [ ]:
class FeatureStore(DataReader):
  def __init__(self, tickers, start, end, cons, reader_params, debug:bool=False):
    super().__init__(tickers, start, end, cons, reader_params)
    self.debug = debug

  def load_or_generate_data(self):
    returns_path = f"returns_{self.start}_{self.end}.parquet"
    factors_path = f"factors_{self.start}_{self.end}.parquet"
    signals_path = f"signals_{self.start}_{self.end}.parquet"

    if os.path.exists(returns_path) and os.path.exists(factors_path) and os.path.exists(signals_path) and not self.debug:
      print("loading data")
      self.signals = pd.read_parquet(signals_path)
      self.returns = pd.read_parquet(returns_path)
      self.factors = pd.read_parquet(factors_path)

    else:
      print("generating data")

      self.generate_data()

      factors = self.datastore["factors"]
      market_data = self.datastore["market_data"]

      self.signals = self.compute_signals()
      self.returns, self.factors = self.construct_factors()
      self.signals.to_parquet(signals_path)
      self.returns.to_parquet(returns_path)
      self.factors.to_parquet(factors_path)


  def compute_signals(self):
    omegas = {}
    raw_signals = {}

    for i in ['size', "BM", "momentum"]:
      x = self.datastore[i]
      coef = -1 if i == 'size' else 1
      raw_signals[i], sigma = Utils.compute_z_scores(x)
      raw_signals[i] *= coef
      omegas[i] = 1/sigma

    raw_signals_df = pd.DataFrame(raw_signals).T
    omegas = pd.Series(omegas)

    signals = raw_signals_df.mul(omegas, axis=0)

    return signals


  def construct_factors(self):
    returns = self.datastore["market_data"].copy()
    factors = self.datastore["factors"].copy()

    rf = factors["RF"]
    factors = factors.drop(columns=["RF"])

    print(rf.index)
    print(returns.index)

    common_dates = returns.index.intersection(rf.index)
    returns = returns.loc[common_dates]
    rf = rf.loc[common_dates]


    returns = returns.sub(rf, axis=0)

    return returns, factors


In [ ]:
class FactorModel:
  def __init__(self, alpha=10.0, window_size=12, solver="auto"):
    self.alpha = alpha
    self.window_size = window_size

  def calculate_window_beta(self, x, y):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)

    model = Ridge(alpha=self.alpha, fit_intercept=True, solver=self.solver)

    model.fit(x_scaled, y)
    beta = model.coef_
    intercept = model.intercept_

    return (intercept, beta)

  def fit(self, X, returns):
    y = returns.copy()

    factors_list = X.columns
    tickers = y.columns

    n_rows = len(y)
    n_cols = len(tickers) * len(factors_list)

    np_betas = np.empty((n_rows, n_cols))
    np_betas[:] = np.nan
    np_alphas = np.empty((n_rows, len(tickers)))
    np_alphas[:] = np.nan


    cols = pd.MultiIndex.from_product([tickers, factors_list], names=['Ticker', 'Factor'])

    for i, ticker in enumerate(tickers):
      y_series = y[ticker]
      for j in range(self.window_size - 1, n_rows):
          x_window = X.iloc[j-self.window_size+1 : j+1]
          y_window = y_series.iloc[j-self.window_size+1 : j+1]

          results = self.calculate_window_beta(x_window, y_window)

          (alpha, beta) = results[0], results[1]

          start_col = i * len(factors_list)
          end_col = start_col + len(factors_list)

          np_betas[j, start_col:end_col] = beta
          np_alphas[start_col:end_col] = alpha

    betas = pd.DataFrame(np_betas, columns=cols, index=y.index).dropna()
    alphas = pd.DataFrame(np_alphas, columns=y.columns).dropna()

    return alphas, betas




In [ ]:
class FactorPremiaModel:
  def __init__(self, alpha=0.5, lam_alpha=0.4):
   self.factors = ["Mkt-RF",	"SMB",	"HML",	"RMW",	"CM"]
   self.alpha = alpha
   self.lam_alpha = lam_alpha

  def normalize(self, df):
    return df.rolling(1).apply(lambda x: Utils.compute_z_scores(x)).dropna()

  def estimate_lambda(self, factors, alpha):
    factor_means = np.mean(factors, axis=1)
    return alpha * factor_means

  def compute_expected_return(self, betas, scores, factor_returns):
    n_rows = betas.shape[0]
    n_cols = betas.shape[1]
    cols = betas.columns

    common_index = scores.index.intersection(betas.index)

    betas_aligned = betas[common_index]
    scores_aligned = Utils.compute_z_scores(scores[common_index])

    ER_i_empty = np.empty((n_rows, n_cols))
    ER_i_empty[:] = np.nan
    ER_i = pd.DataFrame(ER_i_empty, columns=cols, index=betas.index)

    beta_scaled = {}

    for ticker in cols:
      beta_i = betas[ticker]

      lam = self.estimate_lambda(factor_returns, self.lam_alpha)

      beta_scaled[ticker] = Utils.dot_product(beta_i, lam)

    betas_df = self.normalize(pd.DataFrame(beta_scaled))
    scores_df = self.normalize(scores)


    for ticker in cols:
      score_i = scores[ticker]
      beta_i = betas_df[ticker]

      ER_i[ticker] = self.alpha * score_i + (1 - self.alpha) * beta_scaled[ticker]

    return ER_i





In [ ]:
class FactorRiskModel:
  def __init__(self, ):
    pass

In [ ]:
class HighRiskPortfolio():
  def __init__(self, tickers:list, start:str, end:str, constrains:dict[str:float],
               reader_params:tuple[str], ridge_alpha:float=10.0, premia_alpha:float=0.5,
               lam_alpha:float=0.3, window:int=12, save_model: bool = False,
               model_path: str|None=None, debug:bool=False):

    self.dataStore = FeatureStore(
        tickers = tickers,
        start=start,
        end=end,
        cons=constrains,
        reader_params=reader_params,
        debug=debug
    )

    self.model = FactorModel(alpha=ridge_alpha, window=window)
    self.premia_model = FactorModel(alpha=premia_alpha, lam_alpha=lam_alpha)


  def fit(self):
    self.dataStore.load_or_generate_data()

    self.alphas, self.betas = self.model.fit(self.dataStore.factors, self.dataStore.returns)
    self.facor_premia = self.premia_model.fit(self.betas, self.dataStore.signals, self.dataStore.factors.pct_change().dropna())
    return self.alphas, self.betas, self.factor_premia

  def build_portfolios(self):
    factor_covariance = self.dataStore.factors.cov()
    
    






In [ ]:
class PortfolioConstruction:
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    pass